Creating pdfQuery using casandraDB as vectorstore for querying with pdf

In [ ]:
# Installing langchain componet to use.
from langchain_community.vectorstores import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# support for dataset retireval with huggingface
from datasets import load_dataset

# with cassio , the engine powering the astradb integration in LangChain
# also intialising DB connection
# import cassio

from PyPDF2 import PdfReader

import os 
from dotenv import load_dotenv
load_dotenv()

<!-- Cassandra DB ,Setup by secret keys  -->

In [ ]:
ASTRA_DB_APPLICATION_TOKEN=os.getenv("ASTRA_DB_APPLICATION_TOKEN")


ASTRA_DB_ID =os.getenv("ASTRA_DB_ID")


GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [13]:
#provide the path of pdf file 
pdfcontent=PdfReader(r"C:\Users\CloudJournee\Desktop\python\python\langchain_update\GenAI_RAG_Implementation\AWS Basic Assignments.pdf")
pdfcontent.pages

In [14]:
from typing_extensions import Concatenate

raw_text =""
# read text from pdfcontent
for text, page in enumerate(pdfcontent.pages):
    content=page.extract_text()
    if content:
        raw_text += content


print(raw_text)


AWS Basic Assignments -1 
PHASE 1 — IAM (Security first, always)  
Create IAM user and explore AWS Console , Understand IAM roles, policies 
and permissions  
1. Create IAM User & Explore AWS Console  
 
 
2. Understand IAM: Users, Roles, Policies  
Mental Model  
 User → assumes Role → Role has Policies → Policies define Permissions  
User → assumes Role → Role has Policies → Policies define Permissions  
• Policy = JSON rulebook  
• Role = identity AWS services assume  
• User = human or app  
Example  
• Lambda needs S3 access → attach policy to Lambda role , not user  
Document:  
• One AWS -managed policy  
• One inline policy example  
 
AWS managed policy example : AmazonS3ReadOnlyAccess  
This AWS managed policy grants  read-only access to Amazon S3 , allowing users to list 
buckets, read objects, and retrieve metadata, but  not to create, update, or delete 
resources.   
• Policy name : AmazonS3ReadOnlyAccess  
• ARN: arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess  
• Use case

In [15]:
import cassio
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN,database_id=ASTRA_DB_ID)

In [16]:
# Create a LangchainEmbedding with Huggingface
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2776.43it/s]


In [17]:
# create a Langchain vector store backed by AstraDB
astra_vectore_store=Cassandra(
    embedding=embeddings,
    table_name="QA_mini_demo",
    session=None,
    keyspace=None,
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from h11._abnf import chunk_size

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=70,
    length_function = len,
    separators="\n"
)
texts = text_splitter.split_text(raw_text)
text #length of the text


39

In [19]:
texts

['AWS Basic Assignments -1 \nPHASE 1 — IAM (Security first, always)  \nCreate IAM user and explore AWS Console , Understand IAM roles, policies \nand permissions  \n1. Create IAM User & Explore AWS Console  \n \n \n2. Understand IAM: Users, Roles, Policies  \nMental Model  \n User → assumes Role → Role has Policies → Policies define Permissions  \nUser → assumes Role → Role has Policies → Policies define Permissions  \n• Policy = JSON rulebook  \n• Role = identity AWS services assume  \n• User = human or app',
 '• Role = identity AWS services assume  \n• User = human or app  \nExample  \n• Lambda needs S3 access → attach policy to Lambda role , not user  \nDocument:  \n• One AWS -managed policy  \n• One inline policy example  \n \nAWS managed policy example : AmazonS3ReadOnlyAccess  \nThis AWS managed policy grants  read-only access to Amazon S3 , allowing users to list \nbuckets, read objects, and retrieve metadata, but  not to create, update, or delete \nresources.   \n• Policy name 

# Load the Dataset into the Vector Store 

In [21]:
astra_vectore_store.add_texts(texts)

astra_vectore_index=VectorStoreIndexWrapper(vectorstore=astra_vectore_store)

# Run the QA Cycle 

In [22]:
query ="I was hungry in the early morning , i ate dosa at 10am and since then my stomach started to hurt , why so ?"
response = astra_vectore_store.similarity_search_with_score(query=query, k=4)
response

[(Document(id='56d1c4f815064b5d9633decf0c5ea2c5', metadata={}, page_content='Final truth (this is the real insight)  \nYou didn’t “learn Docker” when you ran docker run . \nYou learned Docker when you understood:  \nWhat runs where, who owns what, and why each layer exists.'),
  np.float64(0.5845222380841582)),
 (Document(id='6eeb89cbd11246d1997b5eee203e40d0', metadata={}, page_content='Think of Dockerfile as a recipe, not food.  \n \n3. Docker Image (the frozen artifact)  \nWhen you run docker build : \n• Docker reads the Dockerfile  \n• Executes steps layer by layer  \n• Produces an image  \nAn image is:  \n• Immutable  \n• Read-only  \n• Portable  \n• Versionable  \nThis is your application artifact . \nKey idea:  \nImages don’t run. They exist.  \n \n4. Docker Container (the running instance)  \nWhen you run docker run : \n• Docker creates a container from an image'),
  np.float64(0.5545606735986305)),
 (Document(id='ad1bbee5a1a34d4bb6139ed84115860e', metadata={}, page_content='• D